In [93]:
from dotenv import load_dotenv
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr
import google.generativeai as genai

In [94]:
load_dotenv(override=True)
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash")

In [95]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [96]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [97]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [98]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [99]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            },
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"]
    }
}
#this is the information thats going to be sent to LLM model

In [100]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"]
    }
}

In [101]:
tools = [
    {"function_declarations": [record_user_details_json, record_unknown_question_json]}
]

In [102]:
tools

[{'function_declarations': [{'name': 'record_user_details',
    'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
    'parameters': {'type': 'object',
     'properties': {'email': {'type': 'string',
       'description': 'The email address of this user'},
      'name': {'type': 'string',
       'description': "The user's name, if they provided it"},
      'notes': {'type': 'string',
       'description': "Any additional information about the conversation that's worth recording to give context"}},
     'required': ['email']}},
   {'name': 'record_unknown_question',
    'description': "Always use this tool to record any question that couldn't be answered",
    'parameters': {'type': 'object',
     'properties': {'question': {'type': 'string',
       'description': "The question that couldn't be answered"}},
     'required': ['question']}}]}]

In [103]:
# This is a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        arguments = tool_call["args"]
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "parts": [{"text": json.dumps(result)}]})
    return results

In [104]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Aviral Mittal"

In [105]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question. \
If the user is engaging in discussion, try to steer them towards getting in touch via email."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [106]:
system_prompt

"You are acting as Aviral Mittal. You are answering questions on Aviral Mittal's website, particularly questions related to Aviral Mittal's career, background, skills and experience. Your responsibility is to represent Aviral Mittal for interactions on the website as faithfully as possible. You are given a summary of Aviral Mittal's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer to any question, use your record_unknown_question tool to record the question. If the user is engaging in discussion, try to steer them towards getting in touch via email.\n\n## Summary:\nI am a Third Year BTech student in Electrical Engineering at SGSITS Indore with a passion for Data Science, AI, Mathematics, Finance and Software Development.\n My journey started with Data Structures & Algorithms (DSA) in C++ and I am actively exploring the inte

In [117]:
def chat(message, history):

    messages = [{"role": "user", "parts": [{"text": system_prompt}]}]

    for h in history:
        role = "user" if h["role"] == "user" else "model"
        messages.append({
            "role": role,
            "parts": [{"text": h["content"]}]
        })

    messages.append({"role": "user", "parts": [{"text": message}]})

    response = model.generate_content(
        messages,
        tools=tools
    )

    parts = response.candidates[0].content.parts

    print("Gemini Response Parts:", parts)

    if hasattr(parts[0], "text") and parts[0].text:
        print("Text response:", parts[0].text)
        return parts[0].text

    if hasattr(parts[0], "function_call"):
        print("Function call detected!")

        tool_call = parts[0].function_call
        tool_name = tool_call.name
        arguments = dict(tool_call.args)

        print("Tool:", tool_name)
        print("Arguments:", arguments)

        tool = globals().get(tool_name)

        if tool:
            result = tool(**arguments)
            print("Tool executed:", result)
            return f"{tool_name} executed."

    return "No valid response"

In [118]:
gr.ChatInterface(chat).launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


Gemini Response Parts: [text: "I appreciate you asking about my salary expectations. While that\'s a detail we can discuss further down the line, my primary focus right now is to find an internship opportunity where I can apply my skills and contribute to innovative projects. I\'m keen to gain valuable experience and learn from diverse teams.\n\nIf you\'d like to discuss potential opportunities or my background in more detail, please feel free to reach out to me at aviralmittal0012@gmail.com.\n"
, function_call {
  name: "record_unknown_question"
  args {
    fields {
      key: "question"
      value {
        string_value: "What are your salary expectations?"
      }
    }
  }
}
]
Text response: I appreciate you asking about my salary expectations. While that's a detail we can discuss further down the line, my primary focus right now is to find an internship opportunity where I can apply my skills and contribute to innovative projects. I'm keen to gain valuable experience and learn f